# RL rollout 实时监测

按训练 step 读取 `train.log` 和 `rollouts/<step>.jsonl`。先修改下一格的 `RUN_DIR`，然后依次运行。自动刷新单元格可用 Jupyter 的停止按钮中断。

In [ ]:
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUN_DIR = Path('/path/to/training-run')
REFRESH_SECONDS = 10
LOG_PATH = RUN_DIR / 'train.log'
ROLLOUT_DIR = RUN_DIR / 'rollouts'
print('log:', LOG_PATH)
print('rollouts:', ROLLOUT_DIR)


In [ ]:
import html
import sys
import time
from IPython.display import HTML, clear_output, display

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from rl_diagnostics.live_monitor import build_snapshot, latest_rollouts

def pct(value):
    return '-' if value is None else f'{value:.1%}'

def num(value, digits=4):
    return '-' if value is None else f'{value:.{digits}f}'

def line_svg(rows, key, title, color, width=760, height=170):
    points = [(row['step'], row.get(key)) for row in rows if row.get(key) is not None]
    if not points:
        return f'<div><b>{html.escape(title)}</b><p>暂无数据</p></div>'
    values = [value for _, value in points]
    low, high = min(values), max(values)
    if high == low:
        high = low + 1.0
    left, top, right, bottom = 42, 24, 18, 30
    inner_w, inner_h = width-left-right, height-top-bottom
    max_step = max(step for step, _ in points)
    min_step = min(step for step, _ in points)
    span = max(max_step-min_step, 1)
    coords = []
    for step, value in points:
        x = left + (step-min_step)/span*inner_w
        y = top + (high-value)/(high-low)*inner_h
        coords.append(f'{x:.1f},{y:.1f}')
    return f'''<div class="chart"><b>{html.escape(title)}</b><svg width="{width}" height="{height}" viewBox="0 0 {width} {height}"><line x1="{left}" y1="{top+inner_h}" x2="{width-right}" y2="{top+inner_h}" stroke="#9ca3af"/><line x1="{left}" y1="{top}" x2="{left}" y2="{top+inner_h}" stroke="#9ca3af"/><polyline fill="none" stroke="{color}" stroke-width="2.5" points="{' '.join(coords)}"/><text x="4" y="{top+8}" font-size="11">{high:.3f}</text><text x="4" y="{top+inner_h}" font-size="11">{low:.3f}</text><text x="{left}" y="{height-8}" font-size="11">step {min_step}</text><text x="{width-right-48}" y="{height-8}" font-size="11">step {max_step}</text></svg></div>'''

def render_monitor():
    snapshot = build_snapshot(LOG_PATH, ROLLOUT_DIR)
    rows = snapshot['steps']
    latest = rows[-1] if rows else {}
    cards = [
        ('最新 step', snapshot['latest_step']),
        ('rollout', latest.get('rollouts', 0)),
        ('有效答案', pct(latest.get('valid_ratio'))),
        ('正奖励', pct(latest.get('positive_ratio'))),
        ('混合组', latest.get('mixed_groups', 0)),
        ('全错组', latest.get('all_wrong_groups', 0)),
        ('截断率', pct(latest.get('clip_ratio'))),
        ('grad norm', num(latest.get('grad_norm'))),
    ]
    card_html = ''.join(f'<div class="card"><span>{html.escape(str(k))}</span><strong>{html.escape(str(v))}</strong></div>' for k, v in cards)
    table_rows = ''.join(
        '<tr>' + ''.join(f'<td>{html.escape(str(value))}</td>' for value in [
            row['step'], row['rollouts'], pct(row['valid_ratio']), pct(row['positive_ratio']),
            row['all_correct_groups'], row['all_wrong_groups'], row['mixed_groups'],
            pct(row['clip_ratio']), num(row['entropy']), num(row['kl_loss']),
            num(row['grad_norm']), num(row['advantage_mean'])
        ]) + '</tr>' for row in rows[-20:]
    )
    samples = latest_rollouts(ROLLOUT_DIR, 6)
    sample_html = ''.join(
        f'''<details><summary>reward={record.get('score')} advantage={record.get('advantage', '-')} pred={html.escape(str(record.get('pred', '-')))}</summary><pre>{html.escape(str(record.get('output', '')))}</pre></details>'''
        for record in samples
    ) or '<p>暂无 rollout 文件</p>'
    charts = ''.join([
        line_svg(rows, 'reward_mean', '平均 Reward', '#047857'),
        line_svg(rows, 'clip_ratio', '回答截断率', '#b45309'),
        line_svg(rows, 'kl_loss', 'KL loss', '#7c3aed'),
        line_svg(rows, 'grad_norm', 'Gradient norm', '#0369a1'),
    ])
    display(HTML(f'''<style>.monitor{{font-family:system-ui;color:#17202a}}.cards{{display:grid;grid-template-columns:repeat(4,minmax(120px,1fr));gap:8px}}.card{{border:1px solid #d1d5db;border-radius:6px;padding:10px;background:#f8fafc}}.card span{{display:block;color:#64748b;font-size:12px}}.card strong{{font-size:22px}}table{{border-collapse:collapse;width:100%;font-size:12px;margin-top:14px}}th,td{{border-bottom:1px solid #e5e7eb;padding:6px;text-align:right}}th:first-child,td:first-child{{text-align:left}}.chart{{margin-top:16px}}details{{border-bottom:1px solid #e5e7eb;padding:6px}}pre{{white-space:pre-wrap;max-height:260px;overflow:auto}}</style><div class="monitor"><p>最后刷新：{time.strftime('%Y-%m-%d %H:%M:%S')}，按 step 更新</p><div class="cards">{card_html}</div>{charts}<h3>最近 20 步</h3><table><thead><tr><th>step</th><th>rollout</th><th>有效</th><th>正奖励</th><th>全对组</th><th>全错组</th><th>混合组</th><th>截断</th><th>entropy</th><th>KL</th><th>grad</th><th>adv mean</th></tr></thead><tbody>{table_rows}</tbody></table><h3>最新回答抽样</h3>{sample_html}</div>'''))

render_monitor()

## 自动刷新
下面单元格会持续运行。训练结束或暂时不看时，点击 Jupyter 的停止按钮。

In [ ]:
try:
    while True:
        clear_output(wait=True)
        render_monitor()
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print('监控已停止')